# POC: rigorous mean reversion (BTCUSDT)

**Does not modify** existing notebooks or their leaderboard CSVs (`poc_backtest_leaderboard*.csv`). This notebook writes **new** outputs only, with filenames **`_<RUN_MODE>`** before `.csv` (e.g. `poc_mr_rigorous_train_focused.csv`) so different `RUN_MODE` values do not overwrite each other.

**Goal:** wider grids + train vs holdout split + merge stats to stress-test overfitting vs real edge.

**Protocol:** tune / screen ideas on `TRAIN_MONTHS` only. Use `HOLDOUT_MONTHS` and forward windows to **validate** frozen hypotheses — do not re-pick the "best" parameter on holdout and call it unbiased.

**`RUN_MODE = "focused"` (default):** 6 configs only — narrowed from the previous 36 using older merged + forward CSVs (e.g. `..._fast.csv`): keep **`delta=0.003`** with **`tp/sl=0.003/0.002`** (both VWAP windows × adaptive on/off), plus two **`0.007/0.004`** profiles that stayed strong on forward OOS. Re-run notebooks to refresh CSVs; new exports include **`Sharpe_ann`** (annualized from monthly returns in `poc_backtest_core`).

Grid runs are **sequential** via `run_backtest_yearly`.

In [12]:
import os

from poc_backtest_core import (
    ADAPTIVE_VOL_MULT_MAX,
    ADAPTIVE_VOL_MULT_MIN,
    ADAPTIVE_VOL_WINDOW,
    analytics_full,
    clear_reports,
    mean_strategy_futures,
    run_backtest_yearly,
)

# --- Data (same parquet layout as other POCs) ---
data_folder = "data"
symbol = "BTCUSDT"
year = 2025

# Holdout = OOS months (do not use for parameter hunting in the same workflow)
HOLDOUT_MONTHS = (10, 11, 12)
TRAIN_MONTHS = tuple(m for m in range(1, 13) if m not in HOLDOUT_MONTHS)

# "focused" = 6 stable configs (recommended); "fast" = 3×2×3×2 full small grid; "full" = large search
RUN_MODE = "focused"  # "focused" | "fast" | "full"

# Suffix for output CSV names: `poc_mr_rigorous_train_<RUN_MODE>.csv` so runs do not overwrite each other
MR_CSV_TAG = str(RUN_MODE).replace(" ", "_")


def mr_csv(stem: str) -> str:
    """e.g. stem='poc_mr_rigorous_train' -> 'poc_mr_rigorous_train_focused.csv' when RUN_MODE=='focused'."""
    return f"{stem}_{MR_CSV_TAG}.csv"


_ADAPT_KW = dict(
    vol_window=ADAPTIVE_VOL_WINDOW,
    vol_mult_min=ADAPTIVE_VOL_MULT_MIN,
    vol_mult_max=ADAPTIVE_VOL_MULT_MAX,
)

_sample = os.path.join(data_folder, f"{symbol}-aggTrades-{year}-01.parquet")
if not os.path.isfile(_sample):
    print(f"No data at {_sample!r}")

In [13]:
# Parameter grids (mean_strategy_futures: delta, vwap_window, adaptive TP/SL)
# Focused list: (delta, vwap_window, adaptive_tp_sl, tp, sl) — see intro markdown.
MR_GRID_FOCUSED: list[tuple[float, int, bool, float, float]] = [
    (0.003, 200, False, 0.003, 0.002),
    (0.003, 400, False, 0.003, 0.002),
    (0.003, 400, False, 0.004, 0.002),
    (0.003, 200, False, 0.004, 0.002),
    (0.003, 300, False, 0.003, 0.002),
    (0.003, 400, False, 0.005, 0.002),
    (0.003, 200, False, 0.005, 0.002),
]

if RUN_MODE == "focused":
    n_combo = len(MR_GRID_FOCUSED)
elif RUN_MODE == "fast":
    MR_DELTAS = [0.0015, 0.002, 0.003]
    MR_VWAP_WINDOWS = [200, 400]
    MR_TP_SL = [
        (0.003, 0.002),
        (0.007, 0.004),
        (0.012, 0.006),
    ]
    MR_ADAPTIVE_FLAGS = [False, True]
    n_combo = len(MR_DELTAS) * len(MR_VWAP_WINDOWS) * len(MR_TP_SL) * len(MR_ADAPTIVE_FLAGS)
else:
    MR_DELTAS = [0.0008, 0.001, 0.0015, 0.002, 0.0025, 0.003, 0.0035, 0.004]
    MR_VWAP_WINDOWS = [100, 200, 400, 800, 1600]
    MR_TP_SL = [
        (0.002, 0.0015),
        (0.003, 0.002),
        (0.005, 0.003),
        (0.007, 0.004),
        (0.01, 0.006),
        (0.012, 0.006),
        (0.015, 0.01),
    ]
    MR_ADAPTIVE_FLAGS = [False, True]
    n_combo = len(MR_DELTAS) * len(MR_VWAP_WINDOWS) * len(MR_TP_SL) * len(MR_ADAPTIVE_FLAGS)

print(f"RUN_MODE={RUN_MODE!r}  ->  {n_combo} configs per eval split")

RUN_MODE='focused'  ->  7 configs per eval split


In [14]:
def run_mean_reversion_grid(
    months: tuple[int, ...] | list[int],
    eval_label: str,
    *,
    data_year: int | None = None,
) -> None:
    """Run mean only; fills global yearly_reports (caller should clear_reports first)."""
    y = data_year if data_year is not None else year
    if RUN_MODE == "focused":
        for delta, vwap_w, adaptive, tp, sl in MR_GRID_FOCUSED:
            run_backtest_yearly(
                "mean",
                mean_strategy_futures,
                data_folder=data_folder,
                symbol=symbol,
                year=y,
                months=months,
                eval_label=eval_label,
                delta=delta,
                vwap_window=vwap_w,
                tp=tp,
                sl=sl,
                adaptive_tp_sl=adaptive,
                **_ADAPT_KW,
            )
        return
    for adaptive in MR_ADAPTIVE_FLAGS:
        for vwap_w in MR_VWAP_WINDOWS:
            for delta in MR_DELTAS:
                for tp, sl in MR_TP_SL:
                    run_backtest_yearly(
                        "mean",
                        mean_strategy_futures,
                        data_folder=data_folder,
                        symbol=symbol,
                        year=y,
                        months=months,
                        eval_label=eval_label,
                        delta=delta,
                        vwap_window=vwap_w,
                        tp=tp,
                        sl=sl,
                        adaptive_tp_sl=adaptive,
                        **_ADAPT_KW,
                    )


def run_mr_split_save_csv(months, eval_label: str, csv_name: str, *, data_year: int | None = None) -> None:
    """Sequential grid + leaderboard CSV (same columns as other POC leaderboards)."""
    clear_reports()
    run_mean_reversion_grid(months, eval_label, data_year=data_year)
    analytics_full(csv_name)

## A) Train months only → `poc_mr_rigorous_train_<RUN_MODE>.csv`

Screen candidates here. This file must not overwrite any original leaderboard CSV.

In [15]:
if os.path.isfile(_sample):
    run_mr_split_save_csv(TRAIN_MONTHS, "mr_rigorous_train", mr_csv("poc_mr_rigorous_train"))
else:
    print("Skip run: missing sample data.")

Running [mr_rigorous_train] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 01
Running [mr_rigorous_train] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 02
Running [mr_rigorous_train] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 03
Running [mr_rigorous_train] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 04
Running [mr_rigorous_train] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 05
Running [mr_rigorous_train] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_

## B) Holdout OOS → `poc_mr_rigorous_holdout_<RUN_MODE>.csv`

Same grid on **holdout months** for out-of-sample comparison (merge in the next section).

In [16]:
if os.path.isfile(_sample):
    run_mr_split_save_csv(HOLDOUT_MONTHS, "mr_rigorous_holdout", mr_csv("poc_mr_rigorous_holdout"))
else:
    print("Skip run: missing sample data.")

Running [mr_rigorous_holdout] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 10
Running [mr_rigorous_holdout] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 11
Running [mr_rigorous_holdout] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 12
Running [mr_rigorous_holdout] mean {'delta': 0.003, 'vwap_window': 400, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 10
Running [mr_rigorous_holdout] mean {'delta': 0.003, 'vwap_window': 400, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 11
Running [mr_rigorous_holdout] mean {'delta': 0.003, 'vwap_window': 400

## D) Train vs holdout merge + overfit diagnostics

Reads the **new** CSVs above. Saves `poc_mr_rigorous_merged_train_holdout_<RUN_MODE>.csv`.

In [17]:
import pandas as pd

from poc_mean_reversion_analysis import (
    merge_train_holdout,
    spearman_train_holdout_returns,
    top_k_oos_degradation,
)

train_path = mr_csv("poc_mr_rigorous_train")
hold_path = mr_csv("poc_mr_rigorous_holdout")

if os.path.isfile(train_path) and os.path.isfile(hold_path):
    df_t = pd.read_csv(train_path)
    df_h = pd.read_csv(hold_path)
    merged = merge_train_holdout(df_t, df_h)
    merged.to_csv(mr_csv("poc_mr_rigorous_merged_train_holdout"), index=False)
    rho = spearman_train_holdout_returns(merged)
    print(f"Spearman corr(train return, holdout return) across grid: {rho:.4f}")
    print("(Closer to +1 means ranking is stable; near 0 means noise; negative = reversal.)")
    display(top_k_oos_degradation(merged, k=15))
else:
    print("Run sections A and B first to produce train/holdout CSVs.")

Spearman corr(train return, holdout return) across grid: 0.7143
(Closer to +1 means ranking is stable; near 0 means noise; negative = reversal.)


,delta,vwap_window,adaptive_tp_sl,tp,sl,return_%_train,return_%_holdout,oos_minus_train_pct
0,0.003,200,False,0.003,0.002,18.16,12.20,-5.96
1,0.003,300,False,0.003,0.002,16.22,6.24,-9.98
2,0.003,400,False,0.003,0.002,15.48,4.59,-10.89
3,0.003,200,False,0.005,0.002,13.20,3.79,-9.41
4,0.003,200,False,0.004,0.002,12.03,5.59,-6.44
5,0.003,400,False,0.004,0.002,10.83,5.21,-5.62
6,0.003,400,False,0.005,0.002,9.39,3.42,-5.97


## E) Optional forward OOS (another year / months)

Set `forward_year` and `forward_months`, then run. Output: `poc_mr_rigorous_forward_oos_<RUN_MODE>.csv` only.

**Other coins:** use the same `*-aggTrades-YYYY-MM.parquet` naming under `data_folder/` and change `symbol` + `forward_year` in the cell below. ETH or other pairs are comparable only after you confirm similar data quality and session coverage.

In [18]:
forward_year = 2026
forward_months = tuple(range(1, 4))  # Q1 example; adjust when parquet files exist

_sample_f = os.path.join(
    data_folder, f"{symbol}-aggTrades-{forward_year}-{forward_months[0]:02d}.parquet"
)

if os.path.isfile(_sample_f):
    run_mr_split_save_csv(
        forward_months,
        "mr_rigorous_forward_oos",
        mr_csv("poc_mr_rigorous_forward_oos"),
        data_year=forward_year,
    )
else:
    print(f"Skip forward OOS: no file {_sample_f!r}")

Running [mr_rigorous_forward_oos] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 01
Running [mr_rigorous_forward_oos] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 02
Running [mr_rigorous_forward_oos] mean {'delta': 0.003, 'vwap_window': 200, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 03
Running [mr_rigorous_forward_oos] mean {'delta': 0.003, 'vwap_window': 400, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 01
Running [mr_rigorous_forward_oos] mean {'delta': 0.003, 'vwap_window': 400, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 02
Running [mr_rigorous_forward_oos] mean {'delta': 0

## Metrics in exported CSVs (`build_summary_dataframe`)

- **`Sharpe_ann`**: annualized Sharpe from **monthly** portfolio returns `(mean/std)*sqrt(12)`, risk-free 0. This is the **standard-style** interpretable number for comparing strategies on this POC.
- **`Sharpe_trade_m_avg`**: legacy mean of per-month **per-trade** `mean(pnl/cap)/std(pnl/cap)` — **not** classical Sharpe; kept for comparison only.

## Next steps toward a live-style alpha (short)

1. **Re-run** sections A–D to refresh CSVs with `Sharpe_ann`; pick **1–2 frozen configs** with good train/holdout return and acceptable `maxDD` (do not re-tune on holdout).
2. **Daily Sharpe** (optional upgrade): resample equity to daily bars and use `sqrt(365)` annualization for 24/7 crypto — implement when you add daily equity export from the simulator.
3. **Execution layer**: paper or small size with real order book / latency; add funding for perps; stress slippage above `TOTAL_COST`.
4. **Risk**: max position %, max daily loss, kill-switch — re-check metrics after caps.
5. **Alpha freeze**: document rules + parameters in one place; only then extend to more symbols or features.